# Lab 6 – Training PointPillars on a KITTI subset (frames 000000–000020)

**Before running anything, switch on the GPU:** Runtime → Change runtime type → **T4 GPU** → Save.
If you already ran cells without a GPU, also do Runtime → **Disconnect and delete runtime**, then start again from the top.

Run every cell top to bottom. Take a screenshot of each output marked 📸 for the report. The last cell zips everything into `lab6_outputs.zip`.

Data: upload **kitti.zip** (from the Lab6_pointpillars folder on your PC) anywhere in your Google Drive. If it isn't found there, cell 4 asks you to pick it from your PC instead.


## 1. Check the GPU 📸


In [ ]:
import torch
assert torch.cuda.is_available(), ('No GPU attached. Runtime -> Change runtime type -> T4 GPU, '
                                   'then Runtime -> Disconnect and delete runtime, and run from the top.')
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '|', torch.cuda.get_device_name(0))
!nvidia-smi


## 2. Recreate the PointPillars environment

Colab now runs Python 3.13 with numpy 2, so last week's `numpy<2` pin is dropped (it breaks Colab's OpenCV, which `pre_process_kitti.py` imports).
Open3D is only used by the 3D viewer, so its import is made optional in case it has no Python 3.13 build.


In [ ]:
import os
%cd /content
!rm -rf PointPillars
!git clone -q https://github.com/zhulf0804/PointPillars.git
%cd /content/PointPillars
!pip install -q numba

p = 'pointpillars/utils/__init__.py'
s = open(p).read().replace('from .vis_o3d import vis_pc, vis_img_3d',
                           'try:\n    from .vis_o3d import vis_pc, vis_img_3d\nexcept ImportError:\n    pass')
open(p, 'w').write(s)

os.environ.setdefault('CUDA_HOME', '/usr/local/cuda')
!python setup.py build_ext --inplace 2>&1 | tail -3
!python -c "from pointpillars.ops import Voxelization, nms_cuda; from pointpillars.utils import read_points; import cv2, numpy; print('PointPillars ops OK | numpy', numpy.__version__, '| cv2', cv2.__version__)"


## 3. Trim the ImageSets to frame IDs ≤ 000020 📸


In [ ]:
import os
sets_dir = '/content/PointPillars/pointpillars/dataset/ImageSets'
for name in ['train', 'val', 'test', 'trainval']:
    path = os.path.join(sets_dir, f'{name}.txt')
    ids = [l.strip() for l in open(path) if l.strip()]
    kept = [i for i in ids if int(i) <= 20]
    with open(path, 'w') as f:
        f.write('\n'.join(kept) + '\n')
    print(f'{name}.txt: {len(ids)} -> {len(kept)} ids: {" ".join(kept)}')


## 4. Copy the KITTI data into PointPillars/kitti 📸


In [ ]:
import glob
from google.colab import drive, files
drive.mount('/content/drive')
%cd /content/PointPillars
hits = sorted(glob.glob('/content/drive/MyDrive/**/kitti*.zip', recursive=True))
if hits:
    zip_path = hits[0]
    print('Found', zip_path)
else:
    print('kitti.zip not found in Google Drive - choose it from your PC instead (81 MB):')
    uploaded = files.upload()
    zip_path = '/content/PointPillars/' + list(uploaded)[0]
!unzip -q -o "{zip_path}" -d /content/PointPillars/
!for d in kitti/training/* kitti/testing/*; do echo "$d: $(ls $d | wc -l) files"; done


## 5. Pre-process the data 📸


In [ ]:
%cd /content/PointPillars
!python pre_process_kitti.py --data_root kitti
!ls kitti


## 6. Train with custom parameters 📸

Parameters (all are command-line arguments in `train.py`):
- **Epochs:** `--max_epoch` (default 160)
- **Checkpoint frequency:** `--ckpt_freq_epoch` (default 20)
- **Learning rate:** `--init_lr` (default 0.00025; the OneCycle scheduler peaks at 10 × this value)
- **Output folder:** `--saved_path` (default `pillar_logs`); checkpoints go to `<saved_path>/checkpoints/epoch_N.pth`

Run A uses the default learning rate, run B a 4× higher one, so the report can compare them.


In [ ]:
%cd /content/PointPillars
!python train.py --data_root kitti --max_epoch 80 --ckpt_freq_epoch 10 --init_lr 0.00025 \
    --num_workers 2 --saved_path runs/A_lr0.00025 2>&1 | tee train_A.log


In [ ]:
%cd /content/PointPillars
!python train.py --data_root kitti --max_epoch 80 --ckpt_freq_epoch 10 --init_lr 0.001 \
    --num_workers 2 --saved_path runs/B_lr0.001 2>&1 | tee train_B.log


## 7. Where the trained model is saved 📸


In [ ]:
%cd /content/PointPillars
!ls -la runs/A_lr0.00025/checkpoints runs/B_lr0.001/checkpoints


## 8. Loss curves 📸


In [ ]:
%cd /content/PointPillars
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

runs = {'A: lr 0.00025': 'runs/A_lr0.00025/summary', 'B: lr 0.001': 'runs/B_lr0.001/summary'}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, path in runs.items():
    ea = EventAccumulator(path); ea.Reload()
    for ax, tag in zip(axes, ['train/total_loss', 'val/total_loss']):
        if tag in ea.Tags()['scalars']:
            ev = ea.Scalars(tag)
            ax.plot([e.step for e in ev], [e.value for e in ev], marker='.', label=name)
            print(f'{name} {tag}: first {ev[0].value:.3f}, last {ev[-1].value:.3f}')
for ax, t in zip(axes, ['Training loss', 'Validation loss']):
    ax.set_title(t); ax.set_xlabel('Step'); ax.set_ylabel('Total loss'); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.savefig('loss_curves.png', dpi=120); plt.show()


## 9. Evaluate on the validation frames (KITTI AP) 📸


In [ ]:
%cd /content/PointPillars
import os

# Patch evaluate.py: the model returns ([], [], []) instead of a dict for a frame with no predicted
# boxes (common for a model trained on only 12 frames), which crashes keep_bbox_from_image_range.
with open('evaluate.py') as f:
    code = f.read()
old = "                result_filter = keep_bbox_from_image_range(result, tr_velo_to_cam, r0_rect, P2, image_shape)\n"
new = ("                if isinstance(result, tuple):  # no boxes predicted for this frame\n"
       "                    result = {'lidar_bboxes': np.zeros((0, 7), dtype=np.float32),\n"
       "                              'labels': np.zeros((0,), dtype=np.int64),\n"
       "                              'scores': np.zeros((0,), dtype=np.float32)}\n"
       + old)
if 'no boxes predicted' not in code:
    assert code.count(old) == 1, 'evaluate.py line not found - repo has changed'
    with open('evaluate.py', 'w') as f:
        f.write(code.replace(old, new))
print('evaluate.py patched')

ckpts = {'pretrained': 'pretrained/epoch_160.pth',
         'A_lr0.00025': 'runs/A_lr0.00025/checkpoints/epoch_80.pth',
         'B_lr0.001': 'runs/B_lr0.001/checkpoints/epoch_80.pth'}
for name, ckpt in ckpts.items():
    print('#' * 30, name, '#' * 30)
    if not os.path.exists(ckpt):
        print(f'{ckpt} not found - skipping')
        continue
    !python evaluate.py --data_root kitti --ckpt {ckpt} --saved_path eval/{name} --num_workers 2 > eval_{name}.log 2>&1
    if os.path.exists(f'eval/{name}/eval_results.txt'):
        !sed -n '/Overall/,$p' eval/{name}/eval_results.txt
    else:
        print('Evaluation failed - last lines of the log:')
        !tail -n 15 eval_{name}.log


## 10. Inference on an example frame with your own trained model 📸


In [ ]:
%%writefile infer_lab6.py
# Lab 6 inference: run a checkpoint on one KITTI frame and save a BEV plot with predictions vs ground truth.
# Based on the Lab 5 test_colab.py, plus --save_path/--threshold/--title and centre-rotated boxes.
import argparse
import os
from collections import Counter

import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from matplotlib.transforms import Affine2D

from pointpillars.utils import read_points, read_calib, read_label, \
    keep_bbox_from_image_range, keep_bbox_from_lidar_range, bbox_camera2lidar
from pointpillars.model import PointPillars

CLASSES = {'Pedestrian': 0, 'Cyclist': 1, 'Car': 2}
LABEL2CLASSES = {v: k for k, v in CLASSES.items()}
COLORS = {0: '#1f77b4', 1: '#2ca02c', 2: '#d62728'}


def point_range_filter(pts, point_range=[0, -39.68, -3, 69.12, 39.68, 1]):
    keep = (pts[:, 0] > point_range[0]) & (pts[:, 1] > point_range[1]) & (pts[:, 2] > point_range[2]) & \
           (pts[:, 0] < point_range[3]) & (pts[:, 1] < point_range[4]) & (pts[:, 2] < point_range[5])
    return pts[keep]


def add_box(ax, bbox, **kw):
    # bbox: [x, y, z, l, w, h, yaw]; rotate about the box centre, not its corner
    x, y, l, w, yaw = bbox[0], bbox[1], bbox[3], bbox[4], bbox[6]
    rect = Rectangle((x - l / 2, y - w / 2), l, w, facecolor='none', linewidth=2, **kw)
    rect.set_transform(Affine2D().rotate_around(x, y, yaw) + ax.transData)
    ax.add_patch(rect)


def visualize_bev(pc, bboxes, labels, scores, gt_bboxes, title, save_path):
    fig, ax = plt.subplots(figsize=(12, 12))
    ax.scatter(pc[:, 0], pc[:, 1], s=0.5, c='gray', alpha=0.2)

    for bbox, label, score in zip(bboxes, labels, scores):
        add_box(ax, bbox, edgecolor=COLORS.get(int(label), 'red'))
        ax.text(bbox[0], bbox[1] + 1.5, f'{LABEL2CLASSES[int(label)]} {score:.2f}', fontsize=8,
                color=COLORS.get(int(label), 'red'),
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

    if gt_bboxes is not None:
        for bbox in gt_bboxes:
            add_box(ax, bbox, edgecolor='yellow', linestyle='--')

    legend = [Line2D([0], [0], color='gray', lw=0, marker='o', label='Point Cloud')]
    legend += [Line2D([0], [0], color=COLORS[i], lw=2, label=LABEL2CLASSES[i]) for i in range(3)]
    if gt_bboxes is not None and len(gt_bboxes) > 0:
        legend.append(Line2D([0], [0], color='yellow', lw=2, linestyle='--', label='Ground Truth'))
    ax.legend(handles=legend, loc='upper right')

    cnt = Counter(LABEL2CLASSES[int(l)] for l in labels)
    n_gt = 0 if gt_bboxes is None else len(gt_bboxes)
    ax.text(0.01, 0.99, f"Detected: {cnt.get('Car', 0)} Cars, {cnt.get('Cyclist', 0)} Cyclists, "
                        f"{cnt.get('Pedestrian', 0)} Pedestrians  |  GT objects: {n_gt}",
            transform=ax.transAxes, ha='left', va='top', fontsize=11,
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.9))

    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('equal'); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved {save_path}')


def main(args):
    model = PointPillars(nclasses=len(CLASSES)).cuda()
    model.load_state_dict(torch.load(args.ckpt))
    model.eval()

    pc = point_range_filter(read_points(args.pc_path))
    calib = read_calib(args.calib_path)
    tr_velo_to_cam = calib['Tr_velo_to_cam'].astype(np.float32)
    r0_rect = calib['R0_rect'].astype(np.float32)
    P2 = calib['P2'].astype(np.float32)

    with torch.no_grad():
        result = model(batched_pts=[torch.from_numpy(pc).cuda()], mode='test')[0]
    if isinstance(result, tuple):  # the model returns ([], [], []) when it predicts no boxes
        result = {'lidar_bboxes': np.zeros((0, 7), dtype=np.float32),
                  'labels': np.zeros((0,), dtype=np.int64),
                  'scores': np.zeros((0,), dtype=np.float32)}

    image_shape = (375, 1242)  # KITTI image size
    result = keep_bbox_from_image_range(result, tr_velo_to_cam, r0_rect, P2, image_shape)
    result = keep_bbox_from_lidar_range(result, np.array([0, -40, -3, 70.4, 40, 0.0], dtype=np.float32))
    bboxes, labels, scores = result['lidar_bboxes'], result['labels'], result['scores']
    keep = scores >= args.threshold
    bboxes, labels, scores = bboxes[keep], labels[keep], scores[keep]

    gt = read_label(args.gt_path)
    gt_cls = np.array([CLASSES.get(n, -1) for n in gt['name']])
    sel = gt_cls != -1
    gt_cam = np.concatenate([gt['location'], gt['dimensions'], gt['rotation_y'][:, None]], axis=-1)[sel]
    gt_lidar = bbox_camera2lidar(gt_cam, tr_velo_to_cam, r0_rect)

    print(f'{args.title}: {len(bboxes)} detections (threshold {args.threshold}), {len(gt_lidar)} GT objects')
    for label, score in zip(labels, scores):
        print(f'  {LABEL2CLASSES[int(label)]:<10} {score:.3f}')
    visualize_bev(pc, bboxes, labels, scores, gt_lidar, args.title, args.save_path)


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--ckpt', required=True)
    parser.add_argument('--pc_path', required=True)
    parser.add_argument('--calib_path', required=True)
    parser.add_argument('--gt_path', required=True)
    parser.add_argument('--threshold', type=float, default=0.3)
    parser.add_argument('--title', default='BEV detections')
    parser.add_argument('--save_path', default='bev.png')
    main(parser.parse_args())


In [ ]:
%cd /content/PointPillars
from IPython.display import Image, display
frame = '000008'   # validation frame with 6 cars; try 000015 for pedestrians
common = (f'--pc_path kitti/training/velodyne/{frame}.bin --calib_path kitti/training/calib/{frame}.txt '
          f'--gt_path kitti/training/label_2/{frame}.txt --threshold 0.3')
!python infer_lab6.py --ckpt runs/A_lr0.00025/checkpoints/epoch_80.pth {common} --title "Own model (run A, 80 epochs) - frame {frame}" --save_path bev_own_A_{frame}.png
!python infer_lab6.py --ckpt runs/B_lr0.001/checkpoints/epoch_80.pth {common} --title "Own model (run B, 80 epochs) - frame {frame}" --save_path bev_own_B_{frame}.png
!python infer_lab6.py --ckpt pretrained/epoch_160.pth {common} --title "Pretrained model (full KITTI) - frame {frame}" --save_path bev_pretrained_{frame}.png
for f in [f'bev_own_A_{frame}.png', f'bev_own_B_{frame}.png', f'bev_pretrained_{frame}.png']:
    display(Image(f, width=600))


## 11. Download all outputs


In [ ]:
%cd /content/PointPillars
!zip -q -r lab6_outputs.zip train_A.log train_B.log loss_curves.png bev_*.png eval eval_*.log \
    runs/*/summary runs/A_lr0.00025/checkpoints/epoch_80.pth runs/B_lr0.001/checkpoints/epoch_80.pth infer_lab6.py \
    pointpillars/dataset/ImageSets
!ls -la lab6_outputs.zip
from google.colab import files
files.download('lab6_outputs.zip')
